# FraudWatch

## Import packages

In [ ]:
import joblib
import pandas as pd
import streamlit as st
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

## Load and combine datasets

In [ ]:
# Filepaths
train_path = "fraudTrain.csv"
test_path = "fraudTest.csv"

# Load the files
df_train = pd.read_csv(train_path, index_col=0)
df_test = pd.read_csv(test_path, index_col=0)

# Combine into single dataframe
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# The full dataset contains over 1.8 million records, which would slow down training and
# testing greatly to process in full. For the sake of our assignment, we will be only
# be taking the first 100,000 rows of the test dataset.

# Take a sample of 100,000 rows
df = df.sample(100000)
df.head()

## Data Cleaning


In [ ]:
df["merchant"] = df["merchant"].apply(lambda x: str(x).replace("fraud_", ""))

In [ ]:
# Drop unused text/unique string identifiers to prevent data leakage
drop_cols = ["trans_num", "first", "last", "street", "city", "state", "zip", "dob"]
df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Convert timestamp to temporal features
if "trans_date_trans_time" in df_clean.columns:
    df_clean["trans_date_trans_time"] = pd.to_datetime(
        df_clean["trans_date_trans_time"]
    )
    df_clean["hour"] = df_clean["trans_date_trans_time"].dt.hour
    df_clean["day_of_week"] = df_clean["trans_date_trans_time"].dt.dayofweek
    df_clean.drop(columns=["trans_date_trans_time"], inplace=True)

In [ ]:
# Encode categorical variables like job or gender
label_encoders = {}
categorical_cols = ["merchant", "category", "gender", "job"]

for col in categorical_cols:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        label_encoders[col] = le

df.head()

In [ ]:
# Define Features (X) and Target (y)
X = df_clean.drop(columns=["is_fraud"])
y = df_clean["is_fraud"]

## Test/Train Split


In [ ]:
# Stratification ensures identical fraud ratios in train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
# Feature Scaling for SVC distance calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Calculate ratio for XGBoost class weighting
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

## Model Training

In [ ]:
# SVC requires scaled features and stratified sub-sampling if dataset is massive
# CalibratedClassifierCV enables probability outputs for ROC-AUC scoring & Streamlit confidence meters
print("--- Training Model 1: Support Vector Classifier (SVC) ---")
svc_model = CalibratedClassifierCV(
    estimator=SVC(
        kernel="rbf",
        C=1.0,
        class_weight="balanced",
    ),
    ensemble=False,
)

# For standard execution, fit on scaled training set:
svc_model.fit(X_train_scaled, y_train)

In [ ]:
print("--- Training Model 2: XGBoost Classifier ---")
xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=ratio,
    learning_rate=0.1,
    max_depth=6,
    n_jobs=-1,
)
xgb_model.fit(X_train_scaled, y_train)

## Evaluate Models

In [ ]:
# Function to evaluate each model
def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }

    print(f"---------- {name} Evaluation Report ----------")
    print(classification_report(y_test, y_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return metrics

In [ ]:
svc_results = evaluate_model(
    svc_model, "Support Vector Classifier (SVC)", X_test_scaled, y_test
)
xgb_results = evaluate_model(xgb_model, "XGBoost", X_test_scaled, y_test)

# Display side-by-side comparison
comparison_df = pd.DataFrame([svc_results, xgb_results])
print("---------- MODEL COMPARISON SUMMARY ----------")
print(comparison_df.to_string(index=False))

In [ ]:
# Save artifacts for streamlit app
print("\nSaving trained models and preprocessing artifacts...")
joblib.dump(svc_model, "svc_model.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")